# Introduction to Machine Learning

# Building a Model to Recommend a Suitable Mobile Plan

**Goal:**
- Build a classification model that chooses a suitable new mobile plan for the users.
- Optimize the model by the accuracy metric (the minimum acceptable value is 0.75).
- Check the accuracy on the test set.

**Plan:**
- 1. General information about the data
- 2. Building classification models
    - 2.1 Decision tree
    - 2.2 Random forest
    - 2.3 Logistic regression
    - 2.4 K-nearest neighbors
    - 2.5 Support vector machine
    - 2.6 Checking on the test set
- 3. Sanity check of the model
- 4. Overall conclusion

**Data description:**
Each object in the dataset `users_behavior.csv` is the information about the behavior of one user during a month.
- `calls` - number of calls,
- `minutes` - total duration of calls in minutes,
- `messages` - number of sms messages,
- `mb_used` - internet traffic used in MB,
- `is_ultra` - the plan the user had during the month ("Ultra" - 1, "Smart" - 0).

## 1. General Information About the Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score

Load the data from `users_behavior.csv`, print the first 10 rows for a first look at the data and study the table structure with `info()`: data types, number of rows and columns, missing values.

In [ ]:
# read the data from the .csv file with read_csv()
users = pd.read_csv('users_behavior.csv')
# print the first 10 rows of the table with head()
display(users.head(10))
# study the table structure with info(): data types, numbers of rows and columns, missing values.
users.info()

- The table has 3214 objects. There are no missing values.
- The columns `calls` and `messages` contain integer values. Change their data type to `int64` to save memory.

In [ ]:
users['calls'] = users['calls'].astype('int16')
users['messages'] = users['messages'].astype('int16')

- Look at the numerical distributions of the columns with `describe()`.

In [ ]:
users.describe()

- For all features the median and the mean differ little from each other. There is almost no skew in the data.
- Plot the distributions of the features (histograms and boxplots).

In [ ]:
def show_hists(data, data_label, num_bins, rug=False, log_scale=False):
    """
    input: data - data to plot the histogram for
           data_label - description of the data
           num_bins - number of bins
    output: None
    description: The function plots a histogram and a boxplot for the selected parameter:
                - Density (seaborn's distplot())
                - Boxplot (seaborn's boxplot())
    """
    sns.set()
    f, axes = plt.subplots(1, 2, figsize=(16, 4))
    axes[0].set_title(data_label + ". Density", fontsize=18)
    axes[0].set_ylabel('density', fontsize=15)
    sns.distplot(data, bins=num_bins, rug=rug, ax=axes[0])
    axes[1].set_title(data_label + ". Boxplot", fontsize=18)
    sns.boxplot(data=users, x=data_label, ax=axes[1])
    plt.show()

In [ ]:
for column in users.drop('is_ultra', axis=1).columns:
    show_hists(users[column], column, num_bins=30)

- Note that `messages` is 0 for many objects. This introduces a small left skew in the data.

In [ ]:
zero_mes_rate = round(len(users.loc[users['messages'] == 0]) / len(users) * 100, 2)
print('Share of users who do not send messages: {} %'.format(zero_mes_rate))

- Determine the share of outliers for all parameters.

In [ ]:
def outliers_rate(data, column):
    """
    input: data - data table
           column - feature to calculate the share of outliers for
    output: None
    description: The function calculates and prints the share of outliers of the selected feature in %.
                 The function returns the outlier bound.
    """
    # the quantiles are calculated with quantile()
    # the outlier bound is calculated by the formula [ q.75 + (q.75 - q.25)*1.5 ]
    max_val = (data[column].quantile(0.75) - data[column].quantile(0.25))*1.5 + data[column].quantile(0.75)
    rate = round(len(data.loc[data[column] > max_val]) / len(data) * 100, 2)
    print('Share of outliers in the parameter {}: {} %'.format(column, rate))
    return max_val

In [ ]:
# max_vals stores the outlier bounds of the features
max_vals = []
# calculate the outlier bounds of all features and print the shares of outliers of the separate features
for column in users.drop('is_ultra', axis=1).columns:
    max_vals.append(outliers_rate(users, column))

# calculate the overall share of outliers over all features.
whole_rate = len(users.loc[(users['calls'] > max_vals[0]) | 
                (users['minutes'] > max_vals[1]) | 
                (users['messages'] > max_vals[2]) | 
                (users['mb_used'] > max_vals[3]) 
                          ]) / len(users)
print('Overall share of outliers: {} %'.format(round(whole_rate*100, 2)))

- There are quite a lot of outliers (about 6.5%). They are not removed.

### Conclusions

- The data is loaded and analyzed successfully. The table contains 3214 objects, there are no missing values.
- The data types of `calls` and `messages` are changed to the integer `int64`.
- There is no strong skew in the distributions of the features.
- The overall share of outliers is **6.5%**. The outliers are not removed.

# 2. Building Classification Models

- First, select in the original data:
    - features: `calls`, `minutes`, `messages`, `mb_used`
    - target feature: `is_ultra`

In [ ]:
features = users.drop('is_ultra', axis=1)
target = users['is_ultra']

- Split the data into training, validation and test sets in the proportion 3:1:1 (60% training and 20% each for validation and test).

In [ ]:
# split the data with train_test_split from sklearn
X, X_test, y, y_test = train_test_split(features, target, test_size=0.2, random_state=123)
X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.25, random_state=123)

## 2.1 Decision Tree

- Initialize the decision tree model DecisionTreeClassifier.
- Hyperparameters to tune:
    - `max_depth` - maximum depth of the tree
    - `min_samples_split` - minimum number of objects in a node
    - `min_samples_leaf` - minimum number of objects in a leaf
    - `criterion` ("gini" or "entropy") - function that measures the quality of a split
- The quality criterion is the accuracy `accuracy`.
- In later projects `RandomizedSearchCV` and `GridSearchCV` from sklearn are used to tune the model parameters. Here the search is done manually.

In [ ]:
# best_params and best_accuracy store the best model parameters and the highest accuracy respectively
best_params = []
best_accuracy = 0
# loop over all selected hyperparameters
for max_depth in range(2,10):
    for min_samples_split in range(2,6):
        for min_samples_leaf in range(1,10):
            for criterion in ['gini', 'entropy']:
                # initialize the model with the current hyperparameters
                model_dt_clf = DecisionTreeClassifier(random_state=123,
                                                      max_depth=max_depth,
                                                      min_samples_split=min_samples_split,
                                                      min_samples_leaf=min_samples_leaf,
                                                      criterion=criterion)
                # train the model on the training set
                model_dt_clf.fit(X_train, y_train)
                # get the model predictions on the validation set
                predictions = model_dt_clf.predict(X_valid)
                # calculate the model accuracy with accuracy_score
                accuracy = accuracy_score(y_valid, predictions)
                # if the current accuracy is higher than the previous best value,
                # save the model parameters and the current accuracy
                if accuracy > best_accuracy:
                    best_params = [max_depth, min_samples_split, min_samples_leaf, criterion]
                    best_accuracy = accuracy

In [ ]:
# print the best accuracy and the model parameters.
print('''Best accuracy: {}
Model parameters:
   max_depth = {}
   min_samples_split = {}
   min_samples_leaf = {}
   criterion = {} '''.format(round(best_accuracy, 2),
                             best_params[0],
                             best_params[1],
                             best_params[2],
                             best_params[3]
                            ))

- This model already satisfies the requirements of the task. Try to improve the result with other models.

## 2.2 Random Forest

- Initialize the random forest model RandomForestClassifier.
- Hyperparameters to tune:
    - `n_estimators` - number of trees
    - `max_depth` - maximum depth of the tree
    - `min_samples_split` - minimum number of objects in a node
    - `min_samples_leaf` - minimum number of objects in a leaf
    - `criterion` ("gini" or "entropy") - function that measures the quality of a split
- The quality criterion is the accuracy `accuracy`.

In [ ]:
# best_params and best_accuracy store the best model parameters and the highest accuracy respectively
best_params = []
best_accuracy = 0
# loop over all selected hyperparameters
for n_estimators in (1,10):
    for max_depth in range(2,10):
        for min_samples_split in range(2,10):
            for min_samples_leaf in range(1,8):
                for criterion in ['gini', 'entropy']:
                    # initialize the model with the current hyperparameters
                    model_rf_clf = RandomForestClassifier(random_state=123,
                                                          n_estimators=n_estimators,
                                                          max_depth=max_depth,
                                                          min_samples_split=min_samples_split,
                                                          min_samples_leaf=min_samples_leaf,
                                                          criterion=criterion)
                    # train the model on the training set
                    model_rf_clf.fit(X_train, y_train)
                    # get the model predictions on the validation set
                    predictions = model_rf_clf.predict(X_valid)
                    # calculate the model accuracy with accuracy_score
                    accuracy = accuracy_score(y_valid, predictions)
                    # if the current accuracy is higher than the previous best value,
                    # save the model parameters and the current accuracy
                    if accuracy > best_accuracy:
                        best_params = [n_estimators, max_depth, min_samples_split, min_samples_leaf, criterion]
                        best_accuracy = accuracy

In [ ]:
# print the best accuracy and the model parameters.
print('''Best accuracy: {}
Model parameters:
   n_estimators = {}
   max_depth = {}
   min_samples_split = {}
   min_samples_leaf = {}
   criterion = {} '''.format(round(best_accuracy, 2),
                             best_params[0],
                             best_params[1],
                             best_params[2],
                             best_params[3],
                             best_params[4]
                            ))

- The random forest improved the accuracy by 2 percentage points compared with the decision tree.

## 2.3 Logistic Regression

- Initialize the logistic regression model LogisticRegression.
- Hyperparameter to tune:
    - `C` - inverse of the regularization strength
- The quality criterion is the accuracy `accuracy`.

In [ ]:
# best_params and best_accuracy store the best model parameters and the highest accuracy respectively
best_params = []
best_accuracy = 0

cs = np.linspace(0.01, 10, 1000, endpoint=True)
# loop over all selected hyperparameters
for C in cs:
    # initialize the model with the current hyperparameters
    model_lr_clf = LogisticRegression(random_state=123,
                                      C=C,
                                      max_iter=1000)
    # train the model on the training set
    model_lr_clf.fit(X_train, y_train)
    # get the model predictions on the validation set
    predictions = model_lr_clf.predict(X_valid)
    # calculate the model accuracy with accuracy_score
    accuracy = accuracy_score(y_valid, predictions)
    # if the current accuracy is higher than the previous best value,
    # save the model parameters and the current accuracy
    if accuracy > best_accuracy:
        best_params = [C]
        best_accuracy = accuracy

In [ ]:
# print the best accuracy and the model parameters.
print('''Best accuracy: {}
Model parameters:
   C = {} '''.format(round(best_accuracy, 2),
                             best_params[0]
                            ))

- Logistic regression gives a lower accuracy than the decision tree and the random forest.

## 2.4 K-Nearest Neighbors

- Initialize the k-nearest neighbors model KNeighborsClassifier.
- Hyperparameter to tune:
    - `n_neighbors` - number of neighbors used to assign a class
- The quality criterion is the accuracy `accuracy`.

In [ ]:
# best_params and best_accuracy store the best model parameters and the highest accuracy respectively
best_params = []
best_accuracy = 0

# loop over all selected hyperparameters
for n_neighbors in range(1,51):
    # initialize the model with the current hyperparameters
    model_knn_clf = KNeighborsClassifier(n_neighbors=n_neighbors)
    # train the model on the training set
    model_knn_clf.fit(X_train, y_train)
    # get the model predictions on the validation set
    predictions = model_knn_clf.predict(X_valid)
    # calculate the model accuracy with accuracy_score
    accuracy = accuracy_score(y_valid, predictions)
    # if the current accuracy is higher than the previous best value,
    # save the model parameters and the current accuracy
    if accuracy > best_accuracy:
        best_params = [n_neighbors]
        best_accuracy = accuracy

In [ ]:
# print the best accuracy and the model parameters.
print('''Best accuracy: {}
Model parameters:
   n_neighbors = {} '''.format(round(best_accuracy, 2),
                             best_params[0]
                            ))

- The classification accuracy of this method is lower than that of the random forest.

## 2.5 Support Vector Machine

- Initialize the support vector model SVC.
- Hyperparameters to tune:
    - `kernel` - type of the kernel function ['linear', 'poly', 'rbf', 'sigmoid', 'precomputed']
    - `gamma` - kernel coefficient for ['linear', 'rbf', 'sigmoid']
    - `C` - regularization parameter
    - `degree` - degree of the polynomial for the kernel function 'poly'
- The quality criterion is the accuracy `accuracy`.

In [ ]:
# best_params and best_accuracy store the best model parameters and the highest accuracy respectively
best_params = []
best_accuracy = 0
# loop over all selected hyperparameters
for C in (1,10):
    for kernel in ['linear', 'poly', 'rbf', 'sigmoid']:
        if kernel == 'poly':
            gamma = 'scale'
            for degree in range(2, 5):
                model_svc_clf = SVC(random_state=123,
                                    C=C,
                                    kernel=kernel,
                                    degree=degree,
                                    gamma=gamma)
                model_svc_clf.fit(X_train, y_train)
                predictions = model_svc_clf.predict(X_valid)
                accuracy = accuracy_score(y_valid, predictions)
                if accuracy > best_accuracy:
                    best_params = [C, kernel, degree, gamma]
                    best_accuracy = accuracy
        else:
            degree=3
            for gamma in ['auto', 'scale']:
                model_svc_clf = SVC(random_state=123,
                                    C=C,
                                    kernel=kernel,
                                    degree=degree,
                                    gamma=gamma)
                model_svc_clf.fit(X_train, y_train)
                predictions = model_svc_clf.predict(X_valid)
                accuracy = accuracy_score(y_valid, predictions)
                if accuracy > best_accuracy:
                    best_params = [C, kernel, degree, gamma]
                    best_accuracy = accuracy

In [ ]:
# print the best accuracy and the model parameters.
print('''Best accuracy: {}
Model parameters:
   C = {}
   kernel = {}
   degree = {}
   gamma = {}'''.format(round(best_accuracy, 2),
                             best_params[0],
                             best_params[1],
                             best_params[2],
                             best_params[3]
                            ))

- All hyperparameters of the resulting model except the regularization `C` are the default ones. Try to tune the regularization separately and more precisely.

In [ ]:
best_params = []
best_accuracy = 0
for C in range(300,1302,5):
    model_svc_clf = SVC(random_state=123,
                        C=C)
    model_svc_clf.fit(X_train, y_train)
    predictions = model_svc_clf.predict(X_valid)
    accuracy = accuracy_score(y_valid, predictions)
    if accuracy > best_accuracy:
        best_params = [C]
        best_accuracy = accuracy

In [ ]:
# print the best accuracy and the model parameters.
print('''Best accuracy: {}
Model parameters:
   C = {} '''.format(round(best_accuracy, 2),
                             best_params[0]
                            ))

- The maximum accuracy reached (0.76) is lower than that of the random forest.

## 2.6 Checking the Model on the Test Set

- By the accuracy criterion the best result is reached with the Random Forest model.
- Best accuracy: 0.81
- Model parameters:
    n_estimators = 10  
    max_depth = 9  
    min_samples_split = 6  
    min_samples_leaf = 2  
    criterion = gini  
- Train this model on all available data (training + validation) and calculate the accuracy on the test set.

In [ ]:
model_rf_clf = RandomForestClassifier(random_state=123,
                                      n_estimators=10,
                                      max_depth=9,
                                      min_samples_split=6,
                                      min_samples_leaf=2,
                                      criterion='gini')
model_rf_clf.fit(X, y)
predictions = model_rf_clf.predict(X_test)
accuracy = accuracy_score(y_test, predictions)
print('Model accuracy on the test set: {}'.format(round(accuracy, 2)))

### Conclusions

- Different classification models are analyzed: Decision Tree, Random Forest, Logistic Regression, K-Nearest Neighbors and Support Vector Machine.
- Each model is trained for different combinations of hyperparameters; the best variants are printed for each model.
- The best accuracy is reached by the Random Forest model:
    - Accuracy on the validation set: **0.81**
    - Accuracy on the test set: **0.81**

# 3. Sanity Check of the Model

- For the random forest, look in more detail at how the accuracy depends on the number of trees and the maximum depth of the tree.

In [ ]:
accuracy_df = pd.DataFrame(columns=['n_estimators', 'max_depth', 'accuracy'])
for n_est in range(2, 31):
    for m_depth in range(2, 11):
        model_rf_clf = RandomForestClassifier(random_state=123,
                                      n_estimators=n_est,
                                      max_depth=m_depth)
        model_rf_clf.fit(X, y)
        predictions = model_rf_clf.predict(X_test)
        accuracy = accuracy_score(y_test, predictions)
        accuracy_df = accuracy_df.append({'n_estimators': n_est,
                                          'max_depth': m_depth,
                                          'accuracy': accuracy}, ignore_index=True)
        

In [ ]:
display(accuracy_df.head(10))

- Present the data of the table `accuracy_df` as a pivot table and plot a `heatmap`.

In [ ]:
accuracy_pivot = accuracy_df.pivot("max_depth", "n_estimators", "accuracy")
f, ax = plt.subplots(figsize=(17, 6))
sns.heatmap(accuracy_pivot, annot=True, fmt=".2f", linewidths=.5, ax=ax)
ax.set_title("Accuracy of the 'Random Forest' model", fontsize=18)
plt.show()

- The plot shows that the model reaches one of the maximum accuracy values (**accuracy=0.82**) at **max_depth=6** and **n_estimators=7**. There is no point in increasing these parameters further.

# 4. Overall Conclusion

**1)** The data is loaded and analyzed successfully.
There are no missing values; the data types are converted to match the stored information.
There is no strong skew in the distributions of the features.
The overall share of outliers is **6.5%**. The outliers are not removed.

**2)** Different classification models are analyzed: Decision Tree, Random Forest, Logistic Regression, K-Nearest Neighbors and Support Vector Machine. The models are trained on different combinations of hyperparameters.
- The best accuracy is reached by the Random Forest model:
    - Accuracy on the test set: **0.81**

**3)** The change of the accuracy of the Random Forest model is analyzed for different numbers of trees and maximum tree depths.
- The model reaches the maximum accuracy (**accuracy=0.82**) at **max_depth=6** and **n_estimators=7**. There is no point in increasing these parameters further.